In [1]:
from transformers import AutoProcessor, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model
import torch

In [3]:
import copy

In [2]:
base_model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen3-0.6B", dtype=torch.bfloat16)

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

In [4]:
type(base_model)

transformers.models.qwen3.modeling_qwen3.Qwen3ForCausalLM

In [6]:
peft_config = LoraConfig()

In [7]:
model = get_peft_model(copy.deepcopy(base_model), peft_config)

In [8]:
model2 = get_peft_model(base_model, peft_config)

In [8]:
from transformers import TrainingArguments, HfArgumentParser
from dataclasses import asdict

In [9]:
hf_parser = HfArgumentParser((TrainingArguments))
targs = hf_parser.parse_yaml_file('../t-index/recipes/training_args.yaml')[0]

In [11]:
targs.output_dir = "hi"

In [54]:
targs = TrainingArguments(**asdict(targs))
asdict(targs)

PyTorch: setting up devices


{'output_dir': 't-index/models/qwen3-0.6b_alt-tgemma',
 'per_device_train_batch_size': 16,
 'num_train_epochs': 5,
 'max_steps': -1,
 'learning_rate': '1e-6',
 'lr_scheduler_type': <SchedulerType.COSINE: 'cosine'>,
 'lr_scheduler_kwargs': None,
 'warmup_steps': 0.1,
 'optim': <OptimizerNames.ADAMW_TORCH_FUSED: 'adamw_torch_fused'>,
 'optim_args': None,
 'weight_decay': 0.01,
 'adam_beta1': 0.9,
 'adam_beta2': 0.999,
 'adam_epsilon': 1e-08,
 'optim_target_modules': None,
 'gradient_accumulation_steps': 1,
 'average_tokens_across_devices': True,
 'max_grad_norm': 1.0,
 'label_smoothing_factor': 0.0,
 'bf16': True,
 'fp16': False,
 'bf16_full_eval': False,
 'fp16_full_eval': False,
 'tf32': None,
 'gradient_checkpointing': False,
 'gradient_checkpointing_kwargs': None,
 'torch_compile': False,
 'torch_compile_backend': None,
 'torch_compile_mode': None,
 'use_liger_kernel': False,
 'liger_kernel_config': None,
 'use_cache': False,
 'neftune_noise_alpha': None,
 'torch_empty_cache_steps': 

In [32]:
type(targs.learning_rate)

str

In [2]:
import torch
import torch.nn.functional as F

In [7]:
-F.logsigmoid(torch.tensor([0., 0., 0.], device='cuda'))

tensor([0.6931, 0.6931, 0.6931], device='cuda:0')

In [1]:
import numpy as np

In [10]:
np.max([[1],[3],[5]], 0).item()

5

In [1]:
from transformers import PreTrainedTokenizerBase, XLMRobertaTokenizer, AutoTokenizer, AutoProcessor

In [8]:
tokenizer = AutoTokenizer.from_pretrained("facebook/xlm-roberta-xl", use_fast=False)

In [2]:
processor = AutoProcessor.from_pretrained("aisingapore/Qwen-SEA-LION-v4-8B-VL")

In [7]:
print(processor.chat_template)

{%- if tools %}
    {{- '<|im_start|>system\n' }}
    {%- if messages[0].role == 'system' %}
        {%- if messages[0].content is string %}
            {{- messages[0].content }}
        {%- else %}
            {%- for content in messages[0].content %}
                {%- if 'text' in content %}
                    {{- content.text }}
                {%- endif %}
            {%- endfor %}
        {%- endif %}
        {{- '\n\n' }}
    {%- endif %}
    {{- "# Tools\n\nYou may call one or more functions to assist with the user query.\n\nYou are provided with function signatures within <tools></tools> XML tags:\n<tools>" }}
    {%- for tool in tools %}
        {{- "\n" }}
        {{- tool | tojson }}
    {%- endfor %}
    {{- "\n</tools>\n\nFor each function call, return a json object with function name and arguments within <tool_call></tool_call> XML tags:\n<tool_call>\n{\"name\": <function-name>, \"arguments\": <args-json-object>}\n</tool_call><|im_end|>\n" }}
{%- else %}
    {%- if me